# Evaluating unsupervised results

With no labels there is no accuracy. Yet clustering and dimensionality reduction still need a verdict: is this clustering any good, is $k$ right, is it stable, does it help the next step? This short notebook gives four tools in the order you would reach for them: internal metrics, external metrics when labels exist, stability, and the test that matters most, whether the result improves a supervised model.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from common.datasets import load_digits_data, make_customer_data, make_moons_data

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

Xc, seg = make_customer_data(n_samples=1000, seed=SEED)
Xc_s = StandardScaler().fit_transform(Xc)

## 1. Intuition

Two questions get mixed up. "Is this a good clustering?" asks whether groups are tight and well separated. "Is it a useful clustering?" asks whether the groups help you decide something. A metric can answer the first. Only the task can answer the second. The tools below are ordered from cheap and weak to expensive and meaningful.

## 2. Maths

**Silhouette** (notebook 01): $s=\frac{b-a}{\max(a,b)}$ per point, averaged. Range $-1$ to 1, higher is better.

**Davies-Bouldin**: for each cluster, take the worst ratio of (its spread plus another cluster's spread) to the distance between their centres, then average. Compact, far-apart clusters give small ratios. Lower is better, with 0 the minimum.

**Calinski-Harabasz**: the ratio of between-cluster variance to within-cluster variance, adjusted for the number of points and clusters. Higher is better.

**Adjusted Rand index (ARI)**: when true labels exist. Look at every pair of points. A pair is *agreed* if both labelings put it together or both put it apart. The Rand index is the share of agreed pairs, and ARI subtracts what random labelings would agree on by chance, so that 0 means chance and 1 means identical. It ignores label names, so cluster 0 versus class 3 does not matter. **Normalised mutual information (NMI)** is a related score based on shared information.

All of these are summaries. None of them is a cost function that was optimised, except by accident. They exist to compare candidate solutions on the same data.

## 3. Internal metrics

Score k-means on the customer data for several $k$ with all three internal metrics.

In [ ]:
print(" k  silhouette  davies-bouldin  calinski-harabasz")
for k in range(2, 8):
    lab = KMeans(k, n_init=10, random_state=SEED).fit_predict(Xc_s)
    print(
        f"{k:>2}  {silhouette_score(Xc_s, lab):>10.3f}  {davies_bouldin_score(Xc_s, lab):>14.3f}  {calinski_harabasz_score(Xc_s, lab):>17.0f}"
    )

Silhouette peaks, Davies-Bouldin bottoms out and Calinski-Harabasz peaks at $k=4$, which agrees with how the data were made. Three metrics agreeing is good evidence. On messier data they often disagree, and then the data do not have one clear answer. Calinski-Harabasz in particular tends to favour small $k$ on some data, so do not rely on it alone.

**Where internal metrics mislead.** They reward compact, convex clusters, which is what k-means makes. They will not reward a correct clustering of any other shape.

In [ ]:
Xm_df, ym = make_moons_data(400, noise=0.07, seed=SEED)
Xm = Xm_df.to_numpy()
km = KMeans(2, n_init=10, random_state=SEED).fit_predict(Xm)
db = DBSCAN(eps=0.2, min_samples=5).fit_predict(Xm)
for name, lab in (("k-means", km), ("DBSCAN", db)):
    print(
        f"{name:>8}: silhouette {silhouette_score(Xm, lab):.3f}  ARI vs true moons {adjusted_rand_score(ym, lab):.3f}"
    )

k-means scores a higher silhouette and is wrong, DBSCAN scores lower and recovers both moons. Internal metrics assume a definition of "cluster", and if it is not yours the score is misleading. Look at the clustering too.

## 4. External metrics, when labels exist

Sometimes you have labels but not for the data you care about, or you want to test an algorithm on data where you know the answer. Then ARI and NMI measure agreement without needing the group names to match. Use them to compare methods on a benchmark, not to tune on production data where no labels exist.

In [ ]:
true_k = KMeans(4, n_init=10, random_state=SEED).fit_predict(Xc_s)
for k in (2, 4, 8):
    lab = KMeans(k, n_init=10, random_state=SEED).fit_predict(Xc_s)
    print(
        f"k={k}: ARI {adjusted_rand_score(seg, lab):.3f}  NMI {normalized_mutual_info_score(seg, lab):.3f}"
    )
shuffled = rng.permutation(4)[seg]  # rename the groups
print(
    "ARI after renaming the true groups:",
    round(adjusted_rand_score(shuffled, true_k), 3),
    "(same as before renaming:",
    round(adjusted_rand_score(seg, true_k), 3),
    ")",
)
print("ARI of random labels:", round(adjusted_rand_score(seg, rng.integers(0, 4, len(seg))), 3))

ARI is highest at the true $k=4$. Too few clusters merge groups and too many split them. Renaming the groups changes nothing, and random labels score about 0.

## 5. Stability

If a clustering is real, a different sample of the same data should give the same groups. Fit k-means on many bootstrap resamples, label the full data with each fitted model, and measure the average ARI between pairs of labelings. Near 1 is stable. Low means the result depends on the sample, so the structure is weak or $k$ is wrong.

In [ ]:
def stability(X: np.ndarray, k: int, n_boot: int = 15, seed: int = 0) -> float:
    r = np.random.default_rng(seed)
    labelings = []
    for b in range(n_boot):
        idx = r.choice(len(X), len(X), replace=True)
        model = KMeans(k, n_init=3, random_state=b).fit(X[idx])
        labelings.append(model.predict(X))
    scores = [
        adjusted_rand_score(labelings[i], labelings[j])
        for i in range(n_boot)
        for j in range(i + 1, n_boot)
    ]
    return float(np.mean(scores))


ks = list(range(2, 9))
stab = [stability(Xc_s, k) for k in ks]
plt.plot(ks, stab, marker="o")
plt.xlabel("k")
plt.ylabel("mean pairwise ARI across bootstraps")
plt.show()
print({k: round(s, 3) for k, s in zip(ks, stab)})

Stability is 0.99 at $k=2$ and $k=4$, drops to 0.79 at $k=3$, recovers a little to 0.88 at $k=5$ and falls to 0.68 by $k=8$. At $k=3$ and beyond $k=4$ the algorithm has to split the data in ways the structure does not support, and the split changes with each sample. Notice that $k=2$ is just as stable as $k=4$: merging the four segments into two obvious halves is also a reproducible answer. Stability rules out bad choices of $k$ but cannot choose between two good ones, so combine it with silhouette.

## 6. The test that matters: does it help a supervised model?

Clustering and PCA are often steps in a pipeline. Judge them by what the next model scores in cross-validation. Two rules from `03_classical_ml/00_ml_workflow.ipynb` apply: put the step inside the pipeline so each fold fits it on training data only, and compare against the plain baseline.

On the digits, compare a logistic regression on raw pixels, on PCA components, and on k-means features. `KMeans.transform` gives the distance from each point to each centroid, which is a nonlinear re-description of the data.

In [ ]:
Xd, yd = load_digits_data()
Xd = Xd / 16.0
candidates = {
    "pixels": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "PCA 20": make_pipeline(PCA(20, random_state=SEED), LogisticRegression(max_iter=2000)),
    "PCA 10": make_pipeline(PCA(10, random_state=SEED), LogisticRegression(max_iter=2000)),
    "k-means 40 distances": make_pipeline(
        KMeans(40, n_init=3, random_state=SEED), StandardScaler(), LogisticRegression(max_iter=2000)
    ),
}
for name, model in candidates.items():
    scores = cross_val_score(model, Xd, yd, cv=5)
    print(f"{name:>22}: CV accuracy {scores.mean():.3f} +/- {scores.std():.3f}")

Twenty PCA components match the 64 pixels (0.921 against 0.920), while ten lose three points. The k-means distance features give logistic regression a nonlinear view and score 0.938, about two points better than the pixels, which is roughly one standard deviation across folds, so call the gain modest. (Coates and Ng used this idea for images before deep nets.) Which one is right depends on the data and the model, so the cross-validation decides, not intuition.

## 7. What breaks

- **Metrics that agree with the algorithm are circular.** Tuning $k$ by silhouette for k-means rewards the shapes k-means likes.
- **A good score is not a good segmentation.** Four tidy customer groups that marketing cannot act on are worth nothing. Name each cluster, look at its means, and check it means something to the business.
- **Fitting on all the data before splitting leaks.** If a clustering or PCA feeds a model, fit it inside the cross-validation pipeline (as above). Fitting PCA on the full dataset first lets test rows shape the components.
- **Stability depends on the sampling you choose.** It tells you the structure is not an accident of one sample, not that it is the right structure.
- **Labels may not be ground truth.** ARI measures agreement with labels. If those labels are one of several valid groupings, a low ARI is not necessarily a failure.

The closing test for this section checks that you can predict these behaviours before running them.